# Customer 360

**Business question:** Who are our customers, where are they, and what is each market worth?

This notebook builds a consolidated customer view by joining the customer master table
with transaction history to answer:
- How many customers do we have per country?
- What is the gender split in each market?
- Which countries generate the most revenue per customer?
- How do markets rank against each other?

**Output table:** `{catalog}.{schema}.customer_360`  
**Source tables:** `samples.bakehouse.sales_customers`, `samples.bakehouse.sales_transactions`

In [ ]:
# Widget parameters are passed in by the Lakeflow job at runtime.
# Default values keep the notebook runnable interactively in the dev workspace.
dbutils.widgets.text("catalog", "dev_databricks_analytics")
dbutils.widgets.text("schema", "md")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE SCHEMA {schema}")

print(f"Target: {catalog}.{schema}")

## Step 1 — Explore raw data

Always preview before transforming. We confirm column names, data types, and
check for NULLs in critical fields before writing anything downstream.

In [ ]:
customers = spark.read.table("samples.bakehouse.sales_customers")
print(f"Customers: {customers.count():,} rows")
customers.printSchema()
customers.show(5, truncate=False)

In [ ]:
transactions = spark.read.table("samples.bakehouse.sales_transactions")
print(f"Transactions: {transactions.count():,} rows")
transactions.show(5, truncate=False)

## Step 2 — Geographic and gender breakdown

We aggregate customers by country, computing total count, female/male split,
and female percentage.

`COUNT_IF` keeps this in a single aggregation pass — no self-join or CASE
subquery needed. `NULLIF` in the denominator prevents a divide-by-zero if any
country row has zero customers.

In [ ]:
customer_geo = spark.sql("""
    SELECT
        country,
        COUNT(*)                                              AS total_customers,
        COUNT_IF(LOWER(gender) = 'female')                    AS female_customers,
        COUNT_IF(LOWER(gender) = 'male')                      AS male_customers,
        ROUND(
            COUNT_IF(LOWER(gender) = 'female') * 100.0
            / NULLIF(COUNT(*), 0),
        1)                                                    AS female_pct
    FROM samples.bakehouse.sales_customers
    GROUP BY country
    ORDER BY total_customers DESC
""")
customer_geo.show(truncate=False)

## Step 3 — Revenue per customer by country

Joining customers with transactions lets us compute average spend per customer
per country — a key metric for market prioritisation.

We use a LEFT JOIN so countries with customers but no recorded transactions
still appear (with NULL revenue) rather than being silently dropped from the output.

In [ ]:
revenue_by_country = spark.sql("""
    SELECT
        c.country,
        COUNT(DISTINCT c.customer_id)                         AS total_customers,
        COUNT(t.transaction_id)                               AS total_transactions,
        ROUND(SUM(t.totalPrice), 2)                           AS total_revenue,
        ROUND(AVG(t.totalPrice), 2)                           AS avg_transaction_value,
        ROUND(
            SUM(t.totalPrice) / NULLIF(COUNT(DISTINCT c.customer_id), 0),
        2)                                                    AS revenue_per_customer
    FROM samples.bakehouse.sales_customers  c
    LEFT JOIN samples.bakehouse.sales_transactions t
        ON c.customer_id = t.customer_id
    GROUP BY c.country
    ORDER BY total_revenue DESC
""")
revenue_by_country.show(truncate=False)

## Step 4 — Rank markets with a window function

`DENSE_RANK()` assigns the same rank to tied countries and leaves no gaps in
the sequence — better than `RANK()` when displaying 'Top N markets' because
gaps in rank numbers confuse business stakeholders.

We combine the geographic breakdown and revenue metrics into one output table
so downstream consumers only need to query a single place.

In [ ]:
customer_360 = spark.sql("""
    WITH geo AS (
        SELECT
            country,
            COUNT(*)                                          AS total_customers,
            COUNT_IF(LOWER(gender) = 'female')                AS female_customers,
            ROUND(
                COUNT_IF(LOWER(gender) = 'female') * 100.0
                / NULLIF(COUNT(*), 0),
            1)                                                AS female_pct
        FROM samples.bakehouse.sales_customers
        GROUP BY country
    ),
    rev AS (
        SELECT
            c.country,
            ROUND(SUM(t.totalPrice), 2)                       AS total_revenue,
            ROUND(
                SUM(t.totalPrice) / NULLIF(COUNT(DISTINCT c.customer_id), 0),
            2)                                                AS revenue_per_customer
        FROM samples.bakehouse.sales_customers  c
        LEFT JOIN samples.bakehouse.sales_transactions t
            ON c.customer_id = t.customer_id
        GROUP BY c.country
    )
    SELECT
        g.country,
        g.total_customers,
        g.female_customers,
        g.female_pct,
        r.total_revenue,
        r.revenue_per_customer,
        -- Window function: rank all markets by revenue with no gaps for ties
        DENSE_RANK() OVER (ORDER BY r.total_revenue DESC)     AS revenue_rank
    FROM geo  g
    JOIN rev  r  ON g.country = r.country
    ORDER BY revenue_rank
""")
customer_360.show(truncate=False)

## Step 5 — Write to Unity Catalog

`saveAsTable` with `mode('overwrite')` is idempotent: re-running this notebook
overwrites the previous result cleanly without needing a DROP or TRUNCATE.

In [ ]:
customer_360.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.customer_360")
print(f"Written {customer_360.count()} rows to {catalog}.{schema}.customer_360")